# 02 — Feature Aggregation (Track 1: flat table)

**What this notebook is for.** In the EDA we learned each admission is a
day-by-day table of shape `(n_days, 171)`. Tree models (CatBoost / LightGBM)
can't read a variable-length table — they need **one fixed row per admission**.
This notebook turns each admission into that single row, following the plan in
`feature_design.xlsx`.

**What we build (from the feature design table):**

| group | features |
|---|---|
| DEMO | age, gender, ethnicity |
| ICD  | icd_count (burden) + 91 icd_class_flag |
| MED  | med_count + 41 med_class_flag + 41 med_intensity |
| LAB  | 36 lab_last + 36 lab_class_flag + 36 lab_frac |
| Other| sequence_length (number of days) |

**How to read this notebook.** Every code cell is followed by a *"How to read
this"* note in plain language. We build one block at a time so each piece is
easy to follow.

**Important — we use train + valid only, not test.** Unlike the EDA
(where we only *looked* at train+valid), here we build feature rows for every
labeled admission. We deliberately leave `test` out: its score is not visible
to us during development, so for building the model it effectively does not
exist. We use the provided train/valid split directly (the EDA showed it is
patient-clean and balanced): fit on train, evaluate on valid.

## 1. Load the data

Same loading as the EDA notebook: the three CSVs and the pickle with the
day-by-day features. We also record where each feature group sits inside the
171 columns.

In [1]:
import os, pickle, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

DATA_DIR = "data"
EHR_PICKLE = os.path.join(DATA_DIR, "ehr_preprocessed_seq_by_day_cat_embedding.pkl")
LABEL_COL = "readmitted_within_30days"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid.csv"))
# we do NOT load test.csv — its score isn't visible during development,
# so for building the model it effectively does not exist.

with open(EHR_PICKLE, "rb") as f:
    ehr = pickle.load(f)

feat_dict = ehr["feat_dict"]
demo_cols = ehr["demo_cols"]   # 3 columns
icd_cols  = ehr["icd_cols"]    # 91 columns
lab_cols  = ehr["lab_cols"]    # 36 columns
med_cols  = ehr["med_cols"]    # 41 columns

# where each group sits inside the 171 columns
DEMO = slice(0, 3)
ICD  = slice(3, 94)
LAB  = slice(94, 130)
MED  = slice(130, 171)

print("loaded. admissions with features:", len(feat_dict))

loaded. admissions with features: 14532


**How to read this:** nothing new here — we reloaded the same data and the
same column-group positions from the EDA. **Remember:** `feat_dict[some_id]`
gives the `(n_days, 171)` table for one admission; the slices tell us which
columns are demographics, diagnoses, labs, and meds.

## 2. Build the list of admissions to process

We make one table listing every labeled admission (train + valid), with its
patient id, which split it came from, and its 0/1 label. Every feature we
build will line up with this table, row for row. **Test is intentionally not
included** — see the note at the top.

In [2]:
# turn the label into 0/1 for train and valid (we do NOT use test here)
def to_binary(s):
    return s.map(lambda v: 1 if v in {1, "1", True, "True"} else 0).astype(int)

train_df["y"] = to_binary(train_df[LABEL_COL])
valid_df["y"] = to_binary(valid_df[LABEL_COL])

frames = []
for df, split in [(train_df, "train"), (valid_df, "valid")]:
    part = df.drop_duplicates("id")[["id", "y"]].copy()
    part["split"] = split
    frames.append(part)

admissions = pd.concat(frames, ignore_index=True)
admissions["subject_id"] = admissions["id"].str.split("_").str[0]
admissions = admissions[["id", "subject_id", "split", "y"]].reset_index(drop=True)

print("labeled admissions to process:", len(admissions))
print(admissions["split"].value_counts())
admissions.head()

labeled admissions to process: 11022
split
train    8234
valid    2788
Name: count, dtype: int64


,id,subject_id,split,y
0,10869829_25238191,10869829,train,0
1,12347278_29852086,12347278,train,0
2,11157850_29572307,11157850,train,1
3,11958726_24320913,11958726,train,0
4,11944377_26689168,11944377,train,0


**How to read this:** we now have one row per admission for the train and
valid splits, each with its real 0/1 label. **Remember:** the order of this
table is the order every feature block below will follow, so the pieces can be
glued together safely at the end.

*Small tip:* keep `id`, `subject_id`, and `split` alongside the features — we'll need `subject_id` for patient-grouped work and `split` to separate train from valid again after modeling.

## 3. A small helper to loop over admissions in order

Every block below does the same thing: go through each admission in the order of
the `admissions` table, compute some numbers from its `(n_days, 171)` array, and
stack the results. This tiny helper does that looping so each block's code stays
short and readable.

In [3]:
ids_in_order = admissions["id"].tolist()

def build_features(fn):
    """Apply fn to each admission's (n_days, 171) array, stack results into rows."""
    rows = [fn(feat_dict[i]) for i in ids_in_order]
    return np.vstack(rows)

# quick check that ordering is consistent
print("first admission id:", ids_in_order[0])
print("its table shape   :", feat_dict[ids_in_order[0]].shape)

first admission id: 10869829_25238191
its table shape   : (13, 171)


**How to read this:** `build_features` takes a small function that knows how to
summarize *one* admission, and runs it over *all* admissions in the right order,
returning a 2D array (one row per admission). **Remember:** we'll write one small
`fn` per feature idea below; the helper handles the repetitive looping.

## 4. Demographics: age, gender, ethnicity

Demographics don't change during a stay (we checked in the EDA), so we simply
read them from the first day (`row 0`). Three columns: age (a number), gender
(0/1), ethnicity (a 7-way category code).

In [4]:
demo_arr = build_features(lambda m: m[0, DEMO])   # first-day demographics
demo = pd.DataFrame(demo_arr, columns=["age", "gender", "ethnicity"])

print("demo shape:", demo.shape)
demo.head()

demo shape: (11022, 3)


,age,gender,ethnicity
0,52,1,6
1,62,1,6
2,51,1,5
3,75,0,5
4,45,1,5


**How to read this:** three columns, one row per admission. Age is a real
number; gender is 0/1; ethnicity is a small integer *code* (not an amount — the
number 5 is just a label, not "more" than 3). **Remember:** when modeling, tell
the model that `ethnicity` (and `gender`) are **categorical**, so it doesn't
treat the codes as quantities. Age can stay numeric.

## 5. Diagnoses (ICD): burden count + 91 present/absent flags

Diagnoses are recorded as present/absent per day. For a whole admission we ask:
"did this diagnosis group appear on *any* day?" That's a `max` over days, then
`> 0` to make it a clean 0/1 flag. We build:

- **icd_count** — how many distinct diagnosis groups are present (a "how complex
  is this patient" number).
- **91 icd_class_flag** — one 0/1 column per diagnosis group.

In [5]:
# for one admission: which ICD groups appear on any day? -> 91 flags (0/1)
def icd_flags(m):
    return (m[:, ICD].max(0) > 0).astype(int)

icd_flag_arr = build_features(icd_flags)
icd = pd.DataFrame(icd_flag_arr, columns=[f"icd_flag__{c}" for c in icd_cols])
icd.insert(0, "icd_count", icd_flag_arr.sum(axis=1))   # burden = number of flags on

print("icd shape:", icd.shape, "(1 count column + 91 flag columns)")
print("icd_count summary:")
print(icd["icd_count"].describe().round(2))
icd.head()

icd shape: (11022, 92) (1 count column + 91 flag columns)
icd_count summary:
count    11022.00
mean         0.42
std          0.93
min          0.00
25%          0.00
50%          0.00
75%          0.00
max          8.00
Name: icd_count, dtype: float64


,icd_count,icd_flag__Y90-Y99,icd_flag__G30-G32,icd_flag__O85-O92,icd_flag__C60-C63,icd_flag__F40-F48,icd_flag__M80-M85,icd_flag__R00-R09,icd_flag__J90-J94,icd_flag__A00-A09,icd_flag__E00-E07,icd_flag__F01-F09,icd_flag__F30-F39,icd_flag__H30-H36,icd_flag__D60-D64,icd_flag__N00-N08,icd_flag__F60-F69,icd_flag__I80-I89,icd_flag__I95-I99,icd_flag__N30-N39,icd_flag__K55-K64,icd_flag__F50-F59,icd_flag__R40-R46,icd_flag__J60-J70,icd_flag__N20-N23,icd_flag__I30-I52,icd_flag__R50-R69,icd_flag__B25-B34,icd_flag__C00-C14,icd_flag__D65-D69,...,icd_flag__L80-L99,icd_flag__B50-B64,icd_flag__O60-O77,icd_flag__C7A-C7A,icd_flag__B85-B89,icd_flag__E50-E64,icd_flag__K00-K14,icd_flag__R20-R23,icd_flag__J00-J06,icd_flag__N60-N65,icd_flag__D37-D48,icd_flag__K35-K38,icd_flag__G00-G09,icd_flag__M05-M14,icd_flag__I10-I16,icd_flag__B35-B49,icd_flag__K40-K46,icd_flag__K70-K77,icd_flag__Q00-Q07,icd_flag__E20-E35,icd_flag__J20-J22,icd_flag__A80-A89,icd_flag__B00-B09,icd_flag__J80-J84,icd_flag__G60-G65,icd_flag__D3A-D3A,icd_flag__G10-G14,icd_flag__B90-B94,icd_flag__N40-N53,icd_flag__R90-R94
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**How to read this:** `icd_count` is the number of distinct diagnosis groups
per admission; the 91 flag columns say *which* ones. Most admissions will have a
low count (the block is sparse, as the EDA showed). **Remember:** `icd_count` is
the single most useful ICD feature; the 91 flags mostly help the model pick out a
few specific high-risk diagnoses.

## 6. Medications: count + 41 given/not flags + 41 intensity

Medications are **counts** (how many administrations of a drug class on a day),
not simple flags — we confirmed this in our checks (values went up to hundreds).
So we build three views:

- **med_count** — how many distinct drug classes were ever given.
- **41 med_class_flag** — for each class, was it ever given (0/1).
- **41 med_intensity** — average administrations per day for each class
  (`total over stay ÷ number of days`). We divide by days on purpose, so a long
  stay doesn't inflate the number just by being long.

In [6]:
# for one admission:
def med_flags(m):
    return (m[:, MED].max(0) > 0).astype(int)          # ever given? -> 41 flags

def med_intensity(m):
    n_days = m.shape[0]
    return m[:, MED].sum(0) / n_days                    # avg administrations per day -> 41 values

med_flag_arr = build_features(med_flags)
med_int_arr  = build_features(med_intensity)

med = pd.DataFrame(med_flag_arr, columns=[f"med_flag__{c}" for c in med_cols])
med_intensity_df = pd.DataFrame(med_int_arr, columns=[f"med_intensity__{c}" for c in med_cols])
med = pd.concat([med, med_intensity_df], axis=1)
med.insert(0, "med_count", med_flag_arr.sum(axis=1))    # number of distinct classes given

print("med shape:", med.shape, "(1 count + 41 flags + 41 intensity)")
print("med_count summary:")
print(med["med_count"].describe().round(2))
med.head()

med shape: (11022, 83) (1 count + 41 flags + 41 intensity)
med_count summary:
count    11022.00
mean        10.28
std          3.68
min          0.00
25%          8.00
50%         10.00
75%         13.00
max         29.00
Name: med_count, dtype: float64


,med_count,med_flag__UNCLASSIFIED DRUG PRODUCTS,med_flag__HORMONES,med_flag__THYROID PREPS,med_flag__SMOKING DETERRENTS,med_flag__ANTIFUNGALS,med_flag__DIURETICS,med_flag__DIAGNOSTIC,med_flag__ANTICOAGULANTS,med_flag__ANTIPARKINSON DRUGS,med_flag__EENT PREPS,med_flag__ANALGESICS,med_flag__ANTIPARASITICS,med_flag__CNS DRUGS,med_flag__ANTIHISTAMINES,med_flag__ANTIPLATELET DRUGS,med_flag__PSYCHOTHERAPEUTIC DRUGS,med_flag__ANTIARTHRITICS,med_flag__CONTRACEPTIVES,med_flag__SKIN PREPS,med_flag__ANTINEOPLASTICS,med_flag__ELECT/CALORIC/H2O,med_flag__CARDIOVASCULAR,med_flag__COUGH/COLD PREPARATIONS,med_flag__MUSCLE RELAXANTS,med_flag__ANTIINFECTIVES/MISCELLANEOUS,med_flag__GASTROINTESTINAL,med_flag__IMMUNOSUPPRESSANTS,med_flag__ANTIVIRALS,med_flag__BLOOD,...,med_intensity__ANTIPARASITICS,med_intensity__CNS DRUGS,med_intensity__ANTIHISTAMINES,med_intensity__ANTIPLATELET DRUGS,med_intensity__PSYCHOTHERAPEUTIC DRUGS,med_intensity__ANTIARTHRITICS,med_intensity__CONTRACEPTIVES,med_intensity__SKIN PREPS,med_intensity__ANTINEOPLASTICS,med_intensity__ELECT/CALORIC/H2O,med_intensity__CARDIOVASCULAR,med_intensity__COUGH/COLD PREPARATIONS,med_intensity__MUSCLE RELAXANTS,med_intensity__ANTIINFECTIVES/MISCELLANEOUS,med_intensity__GASTROINTESTINAL,med_intensity__IMMUNOSUPPRESSANTS,med_intensity__ANTIVIRALS,med_intensity__BLOOD,med_intensity__ANTIASTHMATICS,med_intensity__CARDIAC DRUGS,med_intensity__PRE-NATAL VITAMINS,med_intensity__ANESTHETICS,med_intensity__ANTIBIOTICS,med_intensity__ANTIHYPERGLYCEMICS,med_intensity__ANTIINFLAM.TUMOR NECROSIS FACTOR INHIBITING AGENTS,med_intensity__SEDATIVE/HYPNOTICS,med_intensity__ANTIDOTES,med_intensity__AUTONOMIC DRUGS,med_intensity__VITAMINS,med_intensity__BIOLOGICALS
0,13,0,0,0,0,0,1,0,1,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,1,0,0,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,35.0,13.0,0.0,0.0,0.0,7.0,0.0,0.0,3.0,8.0,2.0,0.0,0.0,18.0,4.0,0.0,3.0,0.0,0.0,0.0,0.0
1,10,0,0,0,0,0,1,0,1,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,23.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,8,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,1.0,0.0,9.0,0.0,2.0,0.0,0.0
3,19,0,0,0,0,0,1,1,1,0,0,1,0,0,0,0,1,0,0,1,0,1,1,0,0,0,1,0,0,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,57.0,7.0,0.0,0.0,0.0,11.0,0.0,0.0,1.0,5.0,3.0,0.0,1.0,19.0,3.0,0.0,1.0,1.0,1.0,3.0,0.0
4,11,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,1,0,1,0,0,0,...,0.0,0.0,1.0,0.0,5.0,0.0,0.0,0.0,0.0,16.0,0.0,0.0,2.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


**How to read this:** `med_count` (how many different drug classes) was one of
the strongest signals in the EDA — more drug classes tends to mean a sicker,
more complex patient. The flags say *which* classes; intensity says *how much*
of each per day. **Remember:** intensity is per-day on purpose — a raw total
would secretly measure length of stay instead of dosing.

## 7. Labs: last-day flag + ever-abnormal flag + fraction of days abnormal

Labs are the one block that changes day to day, and the values are 0/1
abnormal-flags (we verified this). Because they move over time, we summarize each
lab three ways:

- **lab_last** — abnormal on the final day? (state at discharge, 0/1)
- **lab_class_flag** — abnormal on *any* day? (0/1)
- **lab_frac** — what fraction of days was it abnormal? (0.0–1.0)

In [7]:
def lab_last(m):
    return m[-1, LAB].astype(int)              # final-day value -> 36 values (0/1)

def lab_ever(m):
    return (m[:, LAB].max(0) > 0).astype(int)  # abnormal on any day -> 36 flags

def lab_frac(m):
    return m[:, LAB].mean(0)                    # fraction of days abnormal -> 36 values

lab_last_arr = build_features(lab_last)
lab_ever_arr = build_features(lab_ever)
lab_frac_arr = build_features(lab_frac)

lab = pd.concat([
    pd.DataFrame(lab_last_arr, columns=[f"lab_last__{c}"  for c in lab_cols]),
    pd.DataFrame(lab_ever_arr, columns=[f"lab_flag__{c}"  for c in lab_cols]),
    pd.DataFrame(lab_frac_arr, columns=[f"lab_frac__{c}"  for c in lab_cols]),
], axis=1)

print("lab shape:", lab.shape, "(36 last + 36 ever + 36 frac = 108 columns)")
lab.head()

lab shape: (11022, 108) (36 last + 36 ever + 36 frac = 108 columns)


,lab_last__Creatinine Blood,lab_last__Sodium Blood,lab_last__pO2 Blood,lab_last__Basophils Other Body Fluid,lab_last__Basophils Pleural,lab_last__Lactate Blood,lab_last__Anion Gap Blood,lab_last__Eosinophils Joint Fluid,lab_last__Hemoglobin Blood,lab_last__Chloride Blood,lab_last__Eosinophils Ascites,lab_last__pH Urine,"lab_last__Calcium, Total Blood",lab_last__Lymphocytes Ascites,lab_last__Eosinophils Other Body Fluid,lab_last__Eosinophils Blood,lab_last__Lymphocytes Blood,lab_last__Basophils Joint Fluid,lab_last__Hematocrit Blood,lab_last__Potassium Blood,lab_last__H Blood,lab_last__Monocytes Blood,lab_last__Eosinophils Pleural,lab_last__Troponin T Blood,lab_last__Neutrophils Blood,lab_last__Bicarbonate Blood,lab_last__Basophils Blood,lab_last__Glucose Blood,lab_last__Basophils Ascites,lab_last__pH Blood,...,lab_frac__Anion Gap Blood,lab_frac__Eosinophils Joint Fluid,lab_frac__Hemoglobin Blood,lab_frac__Chloride Blood,lab_frac__Eosinophils Ascites,lab_frac__pH Urine,"lab_frac__Calcium, Total Blood",lab_frac__Lymphocytes Ascites,lab_frac__Eosinophils Other Body Fluid,lab_frac__Eosinophils Blood,lab_frac__Lymphocytes Blood,lab_frac__Basophils Joint Fluid,lab_frac__Hematocrit Blood,lab_frac__Potassium Blood,lab_frac__H Blood,lab_frac__Monocytes Blood,lab_frac__Eosinophils Pleural,lab_frac__Troponin T Blood,lab_frac__Neutrophils Blood,lab_frac__Bicarbonate Blood,lab_frac__Basophils Blood,lab_frac__Glucose Blood,lab_frac__Basophils Ascites,lab_frac__pH Blood,lab_frac__Platelet Count Blood,lab_frac__Lymphocytes Other Body Fluid,lab_frac__Monocytes Ascites,lab_frac__Lymphocytes Joint Fluid,lab_frac__Lymphocytes Pleural,lab_frac__pCO2 Blood
0,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,...,1.000000,1.0,0.076923,0.923077,1.0,1.0,0.692308,1.0,1.0,0.846154,0.846154,1.0,0.076923,0.923077,1.0,0.846154,1.0,1.00,0.846154,0.846154,1.0,0.692308,1.0,0.846154,0.384615,1.0,1.0,1.0,1.0,0.846154
1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1.000000,1.0,0.076923,1.000000,1.0,1.0,0.230769,1.0,1.0,1.000000,0.923077,1.0,0.076923,0.846154,1.0,1.000000,1.0,1.00,0.923077,0.846154,1.0,0.230769,1.0,1.000000,0.923077,1.0,1.0,1.0,1.0,1.000000
2,1,1,0,1,1,1,1,1,0,0,1,1,0,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,...,1.000000,1.0,0.500000,0.750000,1.0,1.0,0.250000,1.0,1.0,1.000000,1.000000,1.0,0.000000,0.750000,1.0,1.000000,1.0,1.00,1.000000,1.000000,1.0,0.000000,1.0,0.500000,1.000000,1.0,1.0,1.0,1.0,0.500000
3,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,0,1,1,1,1,1,1,1,0,1,1,...,0.875000,1.0,0.025000,0.750000,1.0,1.0,0.425000,1.0,1.0,1.000000,0.950000,1.0,0.025000,0.950000,1.0,1.000000,1.0,0.95,0.950000,0.725000,1.0,0.125000,1.0,0.725000,0.700000,1.0,1.0,1.0,1.0,0.725000
4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,0.961538,1.0,0.269231,1.000000,1.0,1.0,0.500000,1.0,1.0,1.000000,1.000000,1.0,0.269231,1.000000,1.0,1.000000,1.0,1.00,1.000000,0.961538,1.0,0.461538,1.0,0.923077,0.615385,1.0,1.0,1.0,1.0,0.846154


**How to read this:** three views of each lab. `lab_last` is the discharge
snapshot (you reasoned this is likely the most predictive — an abnormal lab at
discharge hints at readmission risk); `lab_flag` catches "abnormal at any point";
`lab_frac` measures how *persistently* abnormal it was. **Remember:** some labs
were the same value for nearly everyone (from the sparsity check) — those columns
will be near-constant here and simply won't help the model, which is fine.

## 8. Other: sequence length (number of days)

The number of days in the stay is itself a feature. It's predictive (longer,
more complicated stays relate to readmission), but it has a heavy tail (a few
very long stays up to 231 days), so keep an eye on it later.

In [8]:
seq_len = np.array([feat_dict[i].shape[0] for i in ids_in_order])
other = pd.DataFrame({"sequence_length": seq_len})

print("sequence_length summary:")
print(other["sequence_length"].describe().round(2))
other.head()

sequence_length summary:
count    11022.00
mean        15.01
std         13.35
min          3.00
25%          8.00
50%         11.00
75%         18.00
max        231.00
Name: sequence_length, dtype: float64


,sequence_length
0,13
1,13
2,4
3,40
4,26


**How to read this:** one column, the length of stay in days. **Remember:** it's
a useful feature but a bit of a "how sick" proxy with a long tail — if it ends up
dominating the model later, consider a log transform. For now we keep it as-is.

## 9. Put all the blocks together

Every block above has one row per admission in the same order, so we glue them
side by side, then attach the id / patient / split / label columns from the
`admissions` table. That gives the final feature table.

In [9]:
features = pd.concat([demo, icd, med, lab, other], axis=1)

# attach the bookkeeping columns (kept separate from the model inputs)
final = pd.concat([admissions[["id", "subject_id", "split", "y"]], features], axis=1)

print("final feature table shape:", final.shape)
n_feature_cols = final.shape[1] - 4   # minus id/subject_id/split/y
print("number of model feature columns:", n_feature_cols)
print("\nbreakdown:")
print("  demo :", demo.shape[1])
print("  icd  :", icd.shape[1])
print("  med  :", med.shape[1])
print("  lab  :", lab.shape[1])
print("  other:", other.shape[1])
final.head()

final feature table shape: (11022, 291)
number of model feature columns: 287

breakdown:
  demo : 3
  icd  : 92
  med  : 83
  lab  : 108
  other: 1


,id,subject_id,split,y,age,gender,ethnicity,icd_count,icd_flag__Y90-Y99,icd_flag__G30-G32,icd_flag__O85-O92,icd_flag__C60-C63,icd_flag__F40-F48,icd_flag__M80-M85,icd_flag__R00-R09,icd_flag__J90-J94,icd_flag__A00-A09,icd_flag__E00-E07,icd_flag__F01-F09,icd_flag__F30-F39,icd_flag__H30-H36,icd_flag__D60-D64,icd_flag__N00-N08,icd_flag__F60-F69,icd_flag__I80-I89,icd_flag__I95-I99,icd_flag__N30-N39,icd_flag__K55-K64,icd_flag__F50-F59,icd_flag__R40-R46,...,lab_frac__Eosinophils Joint Fluid,lab_frac__Hemoglobin Blood,lab_frac__Chloride Blood,lab_frac__Eosinophils Ascites,lab_frac__pH Urine,"lab_frac__Calcium, Total Blood",lab_frac__Lymphocytes Ascites,lab_frac__Eosinophils Other Body Fluid,lab_frac__Eosinophils Blood,lab_frac__Lymphocytes Blood,lab_frac__Basophils Joint Fluid,lab_frac__Hematocrit Blood,lab_frac__Potassium Blood,lab_frac__H Blood,lab_frac__Monocytes Blood,lab_frac__Eosinophils Pleural,lab_frac__Troponin T Blood,lab_frac__Neutrophils Blood,lab_frac__Bicarbonate Blood,lab_frac__Basophils Blood,lab_frac__Glucose Blood,lab_frac__Basophils Ascites,lab_frac__pH Blood,lab_frac__Platelet Count Blood,lab_frac__Lymphocytes Other Body Fluid,lab_frac__Monocytes Ascites,lab_frac__Lymphocytes Joint Fluid,lab_frac__Lymphocytes Pleural,lab_frac__pCO2 Blood,sequence_length
0,10869829_25238191,10869829,train,0,52,1,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1.0,0.076923,0.923077,1.0,1.0,0.692308,1.0,1.0,0.846154,0.846154,1.0,0.076923,0.923077,1.0,0.846154,1.0,1.00,0.846154,0.846154,1.0,0.692308,1.0,0.846154,0.384615,1.0,1.0,1.0,1.0,0.846154,13
1,12347278_29852086,12347278,train,0,62,1,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1.0,0.076923,1.000000,1.0,1.0,0.230769,1.0,1.0,1.000000,0.923077,1.0,0.076923,0.846154,1.0,1.000000,1.0,1.00,0.923077,0.846154,1.0,0.230769,1.0,1.000000,0.923077,1.0,1.0,1.0,1.0,1.000000,13
2,11157850_29572307,11157850,train,1,51,1,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1.0,0.500000,0.750000,1.0,1.0,0.250000,1.0,1.0,1.000000,1.000000,1.0,0.000000,0.750000,1.0,1.000000,1.0,1.00,1.000000,1.000000,1.0,0.000000,1.0,0.500000,1.000000,1.0,1.0,1.0,1.0,0.500000,4
3,11958726_24320913,11958726,train,0,75,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1.0,0.025000,0.750000,1.0,1.0,0.425000,1.0,1.0,1.000000,0.950000,1.0,0.025000,0.950000,1.0,1.000000,1.0,0.95,0.950000,0.725000,1.0,0.125000,1.0,0.725000,0.700000,1.0,1.0,1.0,1.0,0.725000,40
4,11944377_26689168,11944377,train,0,45,1,5,2,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,1.0,0.269231,1.000000,1.0,1.0,0.500000,1.0,1.0,1.000000,1.000000,1.0,0.269231,1.000000,1.0,1.000000,1.0,1.00,1.000000,0.961538,1.0,0.461538,1.0,0.923077,0.615385,1.0,1.0,1.0,1.0,0.846154,26


**How to read this:** the full table — bookkeeping columns
(`id`, `subject_id`, `split`, `y`) followed by all the feature columns. The
breakdown should add up to: 3 + 92 + 83 + 108 + 1 = **287 feature columns**.
**Remember:** `id`/`subject_id`/`split`/`y` are *not* model inputs — they're
there so we can split by patient and separate train from valid later. (The next section removes constant "dead" columns, so the final feature count will be a bit lower than 287.)

## 9b. Drop dead feature columns (decided on train only)

In the EDA we found some columns never vary — they hold the same value for
every admission. A column that is constant tells the model nothing (it can't
help tell patients apart), so we remove those. Two important details:

1. **We do this now, at the very end** — *after* `icd_count` and `med_count`
   were already computed. Those two features are sums across a whole block, so
   they needed every raw column present when they were built. By dropping only
   here (on the finished feature table, where every remaining feature depends
   only on itself), we can't accidentally break an aggregate. A dead column is
   a zero column anyway, which adds nothing to a count.
2. **We decide what is "constant" using the training rows only.** A column
   useless in training is useless to the model, and judging this without
   looking at valid keeps the valid split a fair test. We then drop those same
   columns from both train and valid.

In [10]:
feature_only = final.drop(columns=["id", "subject_id", "split", "y"])

# look only at TRAIN rows to decide which columns are constant
train_mask = final["split"] == "train"
train_feats = feature_only[train_mask]

is_constant = train_feats.nunique() <= 1        # True for columns with one value in train
dead_cols = is_constant[is_constant].index.tolist()

# report the dead columns grouped by block prefix, to cross-check the EDA
def block_of(colname):
    if colname.startswith("icd"): return "icd"
    if colname.startswith("med"): return "med"
    if colname.startswith("lab"): return "lab"
    if colname in ("age", "gender", "ethnicity"): return "demo"
    return "other"

from collections import Counter
print("constant (dead) columns found in train:", len(dead_cols))
print("by block:", dict(Counter(block_of(c) for c in dead_cols)))
print("\nexamples:", dead_cols[:8])

# drop them from the full table (train AND valid), keep bookkeeping + live features
final = final.drop(columns=dead_cols)
print("\nfeature columns before:", feature_only.shape[1],
      "-> after drop:", final.shape[1] - 4)

constant (dead) columns found in train: 40
by block: {'icd': 13, 'med': 2, 'lab': 25}

examples: ['icd_flag__O85-O92', 'icd_flag__N00-N08', 'icd_flag__E70-E88', 'icd_flag__A20-A28', 'icd_flag__M50-M54', 'icd_flag__J30-J39', 'icd_flag__N25-N29', 'icd_flag__Q65-Q79']

feature columns before: 287 -> after drop: 247


## 10. Quick sanity checks before saving

A few cheap checks so we don't save a broken table: no rows lost, no accidental
missing values, and the counts look reasonable.

In [11]:
print("rows match admissions:", len(final) == len(admissions))
print("any missing values? :", bool(final.isna().any().any()))
print("\nlabel counts by split:")
print(final.groupby("split")["y"].value_counts())

# a couple of feature ranges as a spot-check
print("\nage range      :", final["age"].min(), "to", final["age"].max())
print("icd_count range:", final["icd_count"].min(), "to", final["icd_count"].max())
print("med_count range:", final["med_count"].min(), "to", final["med_count"].max())
print("lab_frac range :", round(final.filter(like="lab_frac__").values.min(),3),
      "to", round(final.filter(like="lab_frac__").values.max(),3))

rows match admissions: True
any missing values? : False

label counts by split:
split  y
train  0    6785
       1    1449
valid  0    2307
       1     481
Name: count, dtype: int64

age range      : 18 to 99
icd_count range: 0 to 8
med_count range: 0 to 29
lab_frac range : 0.0 to 1.0


**How to read this:** all checks should read sensibly — same number of rows as
admissions, no missing values, labels present for both splits,
and feature ranges within expected bounds (e.g. `lab_frac` between 0 and 1).
**Remember:** if any check looks off, fix it here before saving — it's much
harder to debug once modeling starts.

## 11. Save the feature table

We save to a single file the modeling notebook can load directly. We use a
`.parquet` file because it preserves data types exactly and loads fast; a CSV
would also work if you prefer to open it in a spreadsheet.

In [12]:
os.makedirs("data", exist_ok=True)
OUT_PATH = os.path.join("data", "features_track1.parquet")
final.to_parquet(OUT_PATH, index=False)
print("saved:", OUT_PATH, "with shape", final.shape)

# (optional) also save a CSV if you want to eyeball it in Excel
# final.to_csv(os.path.join("data", "features_track1.csv"), index=False)

saved: data\features_track1.parquet with shape (11022, 251)


**How to read this:** the feature table is saved and ready. **Next notebook:**
load `features_track1.parquet`, fit on the `train` split and evaluate on the
`valid` split, and train CatBoost / LightGBM — measuring AUROC against the
~0.78 baseline from the EDA. (We don't touch the project's `test` set during
development; its score isn't visible to us.)

*Reminder for modeling:* mark `gender` and `ethnicity` as categorical, and drop
the bookkeeping columns (`id`, `subject_id`, `split`, `y`) from the model inputs.